# Cross-step tx→tx graph — Step 2: hand-engineered pair features + ablation

**Goal.** For each transaction `T` at time `t`, summarize the *incoming* cross-step edges (edges `T_src → T` with `t(T_src) < t`) into a per-tx feature vector. Run RF over an ablation grid and decide whether the cross-step graph carries signal beyond the existing trajectory features.

## Two parallel feature blocks
Step 1 found that genuine `out→in` (money-flow) edges are only 17% of all cross-step edges; the rest are co-occurrence patterns. We compute features for both views.

- **Block A** — full graph (all 4 roles, 445K edges).
- **Block B** — money-flow only (`out→in` edges, 76K edges).

## Per-tx feature template (computed for Block A, then again for Block B)
For each tx `T` at time `t`, looking at incoming edges only:
1. `n_in_edges` — count of incoming cross-step edges.
2. `n_unique_src` — unique source txs (multiple wallets can share between same pair).
3. `dt_min`, `dt_mean`, `dt_max` — Δt distribution.
4. **Decayed source-history** with β=0.2: `Σ exp(-β · Δt)` (recency-weighted edge mass).
5. **Source-label aggregates**: `n_illicit_src`, `n_licit_src`, `frac_illicit_src` (over labeled sources only), and decayed-illicit score `Σ_{illicit src} exp(-β · Δt)`.
6. **Source-feature aggregates**: mean & max of `total_BTC`, `fees`, `num_input_addresses`, `num_output_addresses` over source txs.
7. **Role mix** (Block A only): per-role count `(out→in, out→out, in→in, in→out)`.
8. **Top-source intensity**: max edge multiplicity from any single source tx (a single source tx can produce multiple edges if multiple wallets are shared).

## Causality
- Source-label aggregates use `tx_label[src]`; for unknown sources (`label = -1`) they are excluded from the labeled-source counts but kept in `n_in_edges` / `n_unique_src` (the structural count is still observable).
- All edges have `t(src) < t(dst)` strictly (verified in Step 1). At classification time for `T` at `t`, we only look at incoming edges — by construction `t(src) < t`. No future leakage.
- For a labeled training tx at time `t`, source labels with `t(src) < t` reflect ground-truth that *would* have been available to a fraud team by time `t` (consistent with the rest of the project's contract).

## Ablation grid
| | Features |
|---|---|
| A0 | 108 intrinsic (no-graph baseline) |
| A1 | + 103 phase-1 trajectory features |
| **A2** | + Block A cross-step pair features (full graph) |
| **A3** | + Block B cross-step pair features (money-flow only) |
| **A4** | A1 + Block A + Block B (everything) |

Same temporal split as Phase 1: train `t ≤ 34`, test `t ≥ 35`. Same RF hyperparameters (200 trees, balanced).

In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    classification_report,
)

ROOT = Path.cwd()
while not (ROOT / "transactions_data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
TRANSACTIONS_DIR = ROOT / "transactions_data"
WALLETS_DIR      = ROOT / "actors_data"
assert TRANSACTIONS_DIR.exists() and WALLETS_DIR.exists()
print(f"repo root: {ROOT}")

TRAIN_END    = 34
N_TIME_STEPS = 49
RANDOM_SEED  = 175
MAX_WALLET_DEGREE = 50
DECAY_BETA   = 0.2
np.random.seed(RANDOM_SEED)

repo root: /Users/bryanfang/Library/CloudStorage/OneDrive-HarvardUniversity/School/2025-2026/STAT 175/175_final_project


In [2]:
print("Loading data...")
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.txt")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.txt")
tx_classes_df["label"] = tx_classes_df["class"].map({1:1, 2:0, 3:-1}).astype(np.int8)
tx_id_to_idx = {tid: i for i, tid in enumerate(tx_features_df["txId"].values)}
N_tx = len(tx_features_df)

all_cols = list(tx_features_df.columns)
GRAPH_COLS = [c for c in all_cols if c.startswith("Aggregate_feature")] + [
    "in_txs_degree", "out_txs_degree",
]
META_COLS  = ["txId", "Time step"]
feat_cols_intrinsic = [c for c in all_cols if c not in META_COLS and c not in GRAPH_COLS]
F_INTRINSIC = len(feat_cols_intrinsic)

feat_cols_full_182 = [c for c in all_cols if c not in META_COLS]
F_FULL = len(feat_cols_full_182)

merged = tx_features_df[["txId","Time step"]].merge(tx_classes_df[["txId","label"]], on="txId", how="left")
tx_time  = merged["Time step"].astype(np.int64).values
tx_label = merged["label"].fillna(-1).astype(np.int64).values
tx_X_full      = tx_features_df[feat_cols_full_182].fillna(0).astype(np.float32).values
tx_X_intrinsic = tx_features_df[feat_cols_intrinsic].fillna(0).astype(np.float32).values

# For Phase 1 trajectory features we'll need named columns:
agg_named = ["total_BTC","fees","num_input_addresses","num_output_addresses"]
agg_idxs_full      = [feat_cols_full_182.index(c) for c in agg_named]
total_btc_idx_full = feat_cols_full_182.index("total_BTC")

addr_tx_df = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")
tx_addr_df = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")
wallet_addrs = sorted(set(addr_tx_df["input_address"].unique()) |
                      set(tx_addr_df["output_address"].unique()))
wallet_addr_to_idx = {a:i for i, a in enumerate(wallet_addrs)}
addr_tx_df["w"]  = addr_tx_df["input_address"].map(wallet_addr_to_idx)
addr_tx_df["tx"] = addr_tx_df["txId"].map(tx_id_to_idx)
addr_tx_df = addr_tx_df.dropna(subset=["w","tx"]).astype({"w":"int64","tx":"int64"})
tx_addr_df["w"]  = tx_addr_df["output_address"].map(wallet_addr_to_idx)
tx_addr_df["tx"] = tx_addr_df["txId"].map(tx_id_to_idx)
tx_addr_df = tx_addr_df.dropna(subset=["w","tx"]).astype({"w":"int64","tx":"int64"})

incident_in  = defaultdict(list)
incident_out = defaultdict(list)
for tx, w in zip(addr_tx_df["tx"].values, addr_tx_df["w"].values):
    incident_in[int(tx)].append(int(w))
for tx, w in zip(tx_addr_df["tx"].values, tx_addr_df["w"].values):
    incident_out[int(tx)].append(int(w))

wallet_in_txs  = defaultdict(list)
wallet_out_txs = defaultdict(list)
for tx, w in zip(addr_tx_df["tx"].values, addr_tx_df["w"].values):
    wallet_in_txs[int(w)].append((int(tx), int(tx_time[tx])))
for tx, w in zip(tx_addr_df["tx"].values, tx_addr_df["w"].values):
    wallet_out_txs[int(w)].append((int(tx), int(tx_time[tx])))
for w in wallet_in_txs:  wallet_in_txs[w].sort(key=lambda r: r[1])
for w in wallet_out_txs: wallet_out_txs[w].sort(key=lambda r: r[1])

all_incidence = defaultdict(set)
for w, lst in wallet_in_txs.items():
    for tx, _ in lst: all_incidence[w].add(tx)
for w, lst in wallet_out_txs.items():
    for tx, _ in lst: all_incidence[w].add(tx)

print(f"  N_tx={N_tx:,}  F_INTRINSIC={F_INTRINSIC}  F_FULL={F_FULL}")
print(f"  illicit={int((tx_label==1).sum()):,}  licit={int((tx_label==0).sum()):,}")

Loading data...


  N_tx=203,769  F_INTRINSIC=108  F_FULL=182
  illicit=4,545  licit=42,019


In [3]:
print(f"Building cross-step edges (cap={MAX_WALLET_DEGREE})...")
t0 = time.time()
ROLE_OUT_IN, ROLE_OUT_OUT, ROLE_IN_IN, ROLE_IN_OUT = 0, 1, 2, 3
src, dst, dt_arr, role_arr = [], [], [], []
for w in all_incidence:
    if len(all_incidence[w]) < 2 or len(all_incidence[w]) > MAX_WALLET_DEGREE:
        continue
    events = []
    for tx, t in wallet_in_txs.get(w, []):  events.append((t, tx, 'in'))
    for tx, t in wallet_out_txs.get(w, []): events.append((t, tx, 'out'))
    events.sort(key=lambda r: r[0])
    K = len(events)
    for i in range(K):
        ti, txi, si = events[i]
        for j in range(i+1, K):
            tj, txj, sj = events[j]
            if tj <= ti or txi == txj:
                continue
            if   si == 'out' and sj == 'in':  role = ROLE_OUT_IN
            elif si == 'out' and sj == 'out': role = ROLE_OUT_OUT
            elif si == 'in'  and sj == 'in':  role = ROLE_IN_IN
            else:                              role = ROLE_IN_OUT
            src.append(txi); dst.append(txj)
            dt_arr.append(tj - ti); role_arr.append(role)
src      = np.array(src, dtype=np.int64)
dst      = np.array(dst, dtype=np.int64)
dt_edges = np.array(dt_arr, dtype=np.int32)
role_e   = np.array(role_arr, dtype=np.int8)
E = len(src)
print(f"  E={E:,}  built in {time.time()-t0:.1f}s")

Building cross-step edges (cap=50)...


  E=445,180  built in 1.0s


In [4]:
# Group edges by destination tx using sorted argsort + searchsorted
print("Indexing incoming edges per tx...")
t0 = time.time()
order = np.argsort(dst, kind="stable")
dst_sorted   = dst[order]
src_sorted   = src[order]
dt_sorted    = dt_edges[order]
role_sorted  = role_e[order]
# offsets[j], offsets[j+1] give the slice of incoming edges for tx j
offsets = np.searchsorted(dst_sorted, np.arange(N_tx + 1)).astype(np.int64)
print(f"  indexed in {time.time()-t0:.1f}s")

# Pre-extract source-tx feature columns we'll aggregate
src_total_btc = tx_X_full[:, total_btc_idx_full]
src_agg_feats = tx_X_full[:, agg_idxs_full]   # [N_tx, 4]

BLOCK_A_FEATS = [
    "A_n_in_edges", "A_n_unique_src",
    "A_dt_min", "A_dt_mean", "A_dt_max",
    "A_decayed_mass",
    "A_n_illicit_src", "A_n_licit_src", "A_frac_illicit_src",
    "A_decayed_illicit",
    "A_src_total_BTC_mean", "A_src_total_BTC_max",
    "A_src_fees_mean", "A_src_fees_max",
    "A_src_n_in_addr_mean", "A_src_n_out_addr_mean",
    "A_role_out_in", "A_role_out_out", "A_role_in_in", "A_role_in_out",
    "A_max_src_multiplicity",
]
BLOCK_B_FEATS = [
    "B_n_in_edges", "B_n_unique_src",
    "B_dt_min", "B_dt_mean", "B_dt_max",
    "B_decayed_mass",
    "B_n_illicit_src", "B_n_licit_src", "B_frac_illicit_src",
    "B_decayed_illicit",
    "B_src_total_BTC_mean", "B_src_total_BTC_max",
    "B_src_fees_mean", "B_src_fees_max",
    "B_src_n_in_addr_mean", "B_src_n_out_addr_mean",
    "B_max_src_multiplicity",
]
F_A = len(BLOCK_A_FEATS); F_B = len(BLOCK_B_FEATS)

block_A_X = np.zeros((N_tx, F_A), dtype=np.float32)
block_B_X = np.zeros((N_tx, F_B), dtype=np.float32)

print("Computing per-tx pair features (block A + B)...")
t0 = time.time()
for j in range(N_tx):
    a, b = offsets[j], offsets[j+1]
    if a == b:
        continue   # no incoming edges -> all zeros (incl. dt_min stays 0; OK as sentinel)
    s_arr   = src_sorted[a:b]
    dt_a    = dt_sorted[a:b]
    r_a     = role_sorted[a:b]
    src_lab = tx_label[s_arr]
    decay_w = np.exp(-DECAY_BETA * dt_a.astype(np.float64))
    n_e = b - a
    uniq_src, src_mult = np.unique(s_arr, return_counts=True)
    n_uniq = uniq_src.size
    max_mult = int(src_mult.max())
    illicit_m = (src_lab == 1)
    licit_m   = (src_lab == 0)
    n_ill = int(illicit_m.sum()); n_lic = int(licit_m.sum())
    n_lab = n_ill + n_lic

    src_btc = src_total_btc[s_arr]
    src_af  = src_agg_feats[s_arr]   # [n_e, 4]

    # ── Block A — full ──
    block_A_X[j, 0]  = n_e
    block_A_X[j, 1]  = n_uniq
    block_A_X[j, 2]  = float(dt_a.min())
    block_A_X[j, 3]  = float(dt_a.mean())
    block_A_X[j, 4]  = float(dt_a.max())
    block_A_X[j, 5]  = float(decay_w.sum())
    block_A_X[j, 6]  = n_ill
    block_A_X[j, 7]  = n_lic
    block_A_X[j, 8]  = (n_ill / n_lab) if n_lab > 0 else 0.0
    block_A_X[j, 9]  = float(decay_w[illicit_m].sum()) if n_ill > 0 else 0.0
    block_A_X[j,10]  = float(src_af[:,0].mean()); block_A_X[j,11] = float(src_af[:,0].max())
    block_A_X[j,12]  = float(src_af[:,1].mean()); block_A_X[j,13] = float(src_af[:,1].max())
    block_A_X[j,14]  = float(src_af[:,2].mean()); block_A_X[j,15] = float(src_af[:,3].mean())
    block_A_X[j,16]  = float((r_a == ROLE_OUT_IN ).sum())
    block_A_X[j,17]  = float((r_a == ROLE_OUT_OUT).sum())
    block_A_X[j,18]  = float((r_a == ROLE_IN_IN  ).sum())
    block_A_X[j,19]  = float((r_a == ROLE_IN_OUT ).sum())
    block_A_X[j,20]  = max_mult

    # ── Block B — out→in only ──
    bm = (r_a == ROLE_OUT_IN)
    if not bm.any():
        continue
    s_B   = s_arr[bm]; dt_B = dt_a[bm]; lab_B = src_lab[bm]
    dec_B = np.exp(-DECAY_BETA * dt_B.astype(np.float64))
    af_B  = src_af[bm]
    uniq_B, mult_B = np.unique(s_B, return_counts=True)
    ill_Bm = (lab_B == 1); lic_Bm = (lab_B == 0)
    n_ill_B = int(ill_Bm.sum()); n_lic_B = int(lic_Bm.sum()); n_lab_B = n_ill_B + n_lic_B
    block_B_X[j, 0]  = bm.sum()
    block_B_X[j, 1]  = uniq_B.size
    block_B_X[j, 2]  = float(dt_B.min())
    block_B_X[j, 3]  = float(dt_B.mean())
    block_B_X[j, 4]  = float(dt_B.max())
    block_B_X[j, 5]  = float(dec_B.sum())
    block_B_X[j, 6]  = n_ill_B
    block_B_X[j, 7]  = n_lic_B
    block_B_X[j, 8]  = (n_ill_B / n_lab_B) if n_lab_B > 0 else 0.0
    block_B_X[j, 9]  = float(dec_B[ill_Bm].sum()) if n_ill_B > 0 else 0.0
    block_B_X[j,10]  = float(af_B[:,0].mean()); block_B_X[j,11] = float(af_B[:,0].max())
    block_B_X[j,12]  = float(af_B[:,1].mean()); block_B_X[j,13] = float(af_B[:,1].max())
    block_B_X[j,14]  = float(af_B[:,2].mean()); block_B_X[j,15] = float(af_B[:,3].mean())
    block_B_X[j,16]  = int(mult_B.max())

print(f"  done in {time.time()-t0:.1f}s.  block_A_X={block_A_X.shape}  block_B_X={block_B_X.shape}")

Indexing incoming edges per tx...
  indexed in 0.1s
Computing per-tx pair features (block A + B)...


  done in 2.0s.  block_A_X=(203769, 21)  block_B_X=(203769, 17)


In [5]:
# Phase 1 trajectory features (103) — replicate the exact engineering from
# random_forest/phase1_trajectory_signal.ipynb so we can ablate against them.
MAX_INCIDENT_PER_SIDE = 32
MAX_CO_WALLETS        = 150
RECENCY_SENTINEL      = N_TIME_STEPS * 2
TRAJ_DECAY            = 0.2

all_edges = pd.concat([
    addr_tx_df[["w","tx"]].assign(t=tx_time[addr_tx_df["tx"].values],
                                  tx_lab=tx_label[addr_tx_df["tx"].values]),
    tx_addr_df[["w","tx"]].assign(t=tx_time[tx_addr_df["tx"].values],
                                  tx_lab=tx_label[tx_addr_df["tx"].values]),
], ignore_index=True)
all_edges = all_edges.drop_duplicates(subset=["w","tx"]).sort_values(["w","t"]).reset_index(drop=True)
g = all_edges.groupby("w", sort=False)
wallet_t_arr = {}; wallet_tx_arr = {}; wallet_lab_arr = {}
for w, sub in g:
    wallet_t_arr[int(w)]   = sub["t"].values.astype(np.int64)
    wallet_tx_arr[int(w)]  = sub["tx"].values.astype(np.int64)
    wallet_lab_arr[int(w)] = sub["tx_lab"].values.astype(np.int64)
wallet_event_count = {w: len(t) for w, t in wallet_t_arr.items()}
wallet_has_illicit_by = {}
for w, labs in wallet_lab_arr.items():
    m = (labs == 1)
    if m.any():
        wallet_has_illicit_by[w] = int(wallet_t_arr[w][m].min())

def pick_top_wallets(wlist, k=MAX_INCIDENT_PER_SIDE):
    if len(wlist) <= k: return list(wlist)
    cnts = np.array([wallet_event_count.get(w, 0) for w in wlist], dtype=np.int64)
    order = np.argsort(-cnts, kind="stable")
    return [wlist[i] for i in order[:k]]

def per_wallet_priors(w, t_query):
    tarr = wallet_t_arr.get(w)
    if tarr is None: return None
    cut = int(np.searchsorted(tarr, t_query, side="left"))
    if cut == 0: return None
    prior_t   = tarr[:cut]
    prior_lab = wallet_lab_arr[w][:cut]
    prior_tx  = wallet_tx_arr[w][:cut]
    illicit_mask = (prior_lab == 1)
    n_illicit = int(illicit_mask.sum())
    n_licit   = int((prior_lab == 0).sum())
    last_illicit_t = int(prior_t[illicit_mask].max()) if n_illicit > 0 else -1
    if n_illicit > 0:
        decay_w = np.exp(-TRAJ_DECAY * (t_query - prior_t[illicit_mask]).astype(np.float64))
        decayed_illicit_score = float(decay_w.sum())
    else:
        decayed_illicit_score = 0.0
    illicit_frac = n_illicit / max(n_illicit + n_licit, 1)
    co_wallets = set()
    for tx_i in prior_tx:
        for co_w in incident_in.get(int(tx_i), []):
            if co_w != w: co_wallets.add(co_w)
        for co_w in incident_out.get(int(tx_i), []):
            if co_w != w: co_wallets.add(co_w)
        if len(co_wallets) >= MAX_CO_WALLETS: break
    n_co_wallets = len(co_wallets)
    n_co_illicit = sum(1 for cw in co_wallets if wallet_has_illicit_by.get(cw, N_TIME_STEPS+1) < t_query)
    unique_in_partners, unique_out_partners = set(), set()
    for tx_i in prior_tx:
        for co_w in incident_in.get(int(tx_i), []):
            if co_w != w: unique_in_partners.add(co_w)
        for co_w in incident_out.get(int(tx_i), []):
            if co_w != w: unique_out_partners.add(co_w)
    n_in_p = len(unique_in_partners); n_out_p = len(unique_out_partners)
    fan_asymmetry = n_out_p / max(n_in_p, 1)
    age      = int(t_query - prior_t[0])
    recency  = int(t_query - prior_t[-1])
    n_recent_5 = int(((t_query - prior_t) <= 5).sum())
    n_recent_1 = int(((t_query - prior_t) <= 1).sum())
    if cut >= 2:
        iat = np.diff(prior_t.astype(np.float64))
        iat_mean = float(iat.mean()); iat_std = float(iat.std())
        burstiness = float((iat_std - iat_mean) / (iat_std + iat_mean + 1e-8))
    else:
        iat_mean = 0.0; iat_std = 0.0; burstiness = 0.0
    velocity = n_recent_5 / max(cut, 1)
    feat_vals = tx_X_full[prior_tx][:, agg_idxs_full]
    return {
        "n":int(cut),"n_illicit":n_illicit,"n_licit":n_licit,"illicit_frac":illicit_frac,
        "last_illicit_t":last_illicit_t,"decayed_illicit":decayed_illicit_score,
        "n_co_wallets":n_co_wallets,"n_co_illicit":n_co_illicit,
        "co_illicit_frac":n_co_illicit/max(n_co_wallets,1),
        "n_in_partners":n_in_p,"n_out_partners":n_out_p,"fan_asymmetry":fan_asymmetry,
        "first_seen_t":int(prior_t[0]),"last_seen_t":int(prior_t[-1]),
        "age":age,"recency":recency,
        "n_recent_5":n_recent_5,"n_recent_1":n_recent_1,
        "iat_mean":iat_mean,"iat_std":iat_std,"burstiness":burstiness,"velocity":velocity,
        "feat_means":feat_vals.mean(axis=0),"feat_maxes":feat_vals.max(axis=0),
    }

def aggregate_side(summaries, p, t_T):
    n_total = len(summaries); valid = [s for s in summaries if s is not None]; n_w_prior = len(valid)
    out = {f"{p}_n_wallets":n_total, f"{p}_n_wallets_with_prior":n_w_prior,
           f"{p}_frac_first_appearance": (n_total - n_w_prior)/max(n_total,1)}
    keys_init_zero = [
        "n_priors_sum","n_priors_max","n_illicit_sum","n_illicit_max","n_licit_sum",
        "n_wallets_with_illicit","n_wallets_illicit_frac_gt0","n_wallets_illicit_frac_gt50",
        "illicit_frac_max","illicit_frac_mean","decayed_illicit_max","decayed_illicit_sum",
        "recency_to_illicit_min","co_illicit_sum","co_illicit_max","co_illicit_frac_max",
        "co_illicit_frac_mean","n_co_wallets_sum",
        "fan_asymmetry_max","fan_asymmetry_mean","n_in_partners_max","n_out_partners_max",
        "frac_single_use","age_max","age_mean","recency_min","n_recent_5_sum","n_recent_5_max",
        "n_recent_1_sum","velocity_max","velocity_mean","burstiness_max","burstiness_mean",
        "iat_mean_min","iat_std_max",
    ]
    for k in keys_init_zero: out[f"{p}_{k}"] = 0.0
    out[f"{p}_recency_to_illicit_min"] = float(RECENCY_SENTINEL)
    out[f"{p}_recency_min"]            = float(RECENCY_SENTINEL)
    for nm in agg_named:
        out[f"{p}_prior_{nm}_mean_max"] = 0.0
        out[f"{p}_prior_{nm}_max_max"]  = 0.0
    if not valid: return out
    ns = np.array([s["n"] for s in valid], dtype=np.float64)
    nis = np.array([s["n_illicit"] for s in valid], dtype=np.float64)
    nls = np.array([s["n_licit"]   for s in valid], dtype=np.float64)
    ill_frac = np.array([s["illicit_frac"] for s in valid], dtype=np.float64)
    dec_ill  = np.array([s["decayed_illicit"] for s in valid], dtype=np.float64)
    last_ill = np.array([s["last_illicit_t"]  for s in valid], dtype=np.int64)
    has_ill = (last_ill >= 0)
    rec_ill = np.where(has_ill, t_T - last_ill, RECENCY_SENTINEL).astype(np.float64)
    co_ill = np.array([s["n_co_illicit"]    for s in valid], dtype=np.float64)
    co_n   = np.array([s["n_co_wallets"]    for s in valid], dtype=np.float64)
    co_fr  = np.array([s["co_illicit_frac"] for s in valid], dtype=np.float64)
    fan_a  = np.array([s["fan_asymmetry"]   for s in valid], dtype=np.float64)
    n_inp  = np.array([s["n_in_partners"]   for s in valid], dtype=np.float64)
    n_outp = np.array([s["n_out_partners"]  for s in valid], dtype=np.float64)
    ages = np.array([s["age"] for s in valid], dtype=np.float64)
    recs = np.array([s["recency"] for s in valid], dtype=np.float64)
    nr5  = np.array([s["n_recent_5"] for s in valid], dtype=np.float64)
    nr1  = np.array([s["n_recent_1"] for s in valid], dtype=np.float64)
    vel  = np.array([s["velocity"] for s in valid], dtype=np.float64)
    bur  = np.array([s["burstiness"] for s in valid], dtype=np.float64)
    iam  = np.array([s["iat_mean"] for s in valid], dtype=np.float64)
    ias  = np.array([s["iat_std"]  for s in valid], dtype=np.float64)
    fm   = np.stack([s["feat_means"] for s in valid], axis=0)
    fx   = np.stack([s["feat_maxes"] for s in valid], axis=0)
    out[f"{p}_n_priors_sum"] = float(ns.sum()); out[f"{p}_n_priors_max"] = float(ns.max())
    out[f"{p}_n_illicit_sum"] = float(nis.sum()); out[f"{p}_n_illicit_max"] = float(nis.max())
    out[f"{p}_n_licit_sum"]   = float(nls.sum())
    out[f"{p}_n_wallets_with_illicit"] = float(has_ill.sum())
    out[f"{p}_n_wallets_illicit_frac_gt0"]  = float((ill_frac > 0).sum())
    out[f"{p}_n_wallets_illicit_frac_gt50"] = float((ill_frac > 0.5).sum())
    out[f"{p}_illicit_frac_max"]   = float(ill_frac.max())
    out[f"{p}_illicit_frac_mean"]  = float(ill_frac.mean())
    out[f"{p}_decayed_illicit_max"] = float(dec_ill.max())
    out[f"{p}_decayed_illicit_sum"] = float(dec_ill.sum())
    out[f"{p}_recency_to_illicit_min"] = float(rec_ill.min())
    out[f"{p}_co_illicit_sum"] = float(co_ill.sum()); out[f"{p}_co_illicit_max"] = float(co_ill.max())
    out[f"{p}_co_illicit_frac_max"]  = float(co_fr.max())
    out[f"{p}_co_illicit_frac_mean"] = float(co_fr.mean())
    out[f"{p}_n_co_wallets_sum"] = float(co_n.sum())
    out[f"{p}_fan_asymmetry_max"]  = float(fan_a.max())
    out[f"{p}_fan_asymmetry_mean"] = float(fan_a.mean())
    out[f"{p}_n_in_partners_max"]  = float(n_inp.max())
    out[f"{p}_n_out_partners_max"] = float(n_outp.max())
    out[f"{p}_frac_single_use"] = sum(1 for s in valid if s["n"]==1)/max(n_w_prior,1)
    out[f"{p}_age_max"] = float(ages.max()); out[f"{p}_age_mean"] = float(ages.mean())
    out[f"{p}_recency_min"] = float(recs.min())
    out[f"{p}_n_recent_5_sum"] = float(nr5.sum()); out[f"{p}_n_recent_5_max"] = float(nr5.max())
    out[f"{p}_n_recent_1_sum"] = float(nr1.sum())
    out[f"{p}_velocity_max"]  = float(vel.max());  out[f"{p}_velocity_mean"]  = float(vel.mean())
    out[f"{p}_burstiness_max"] = float(bur.max()); out[f"{p}_burstiness_mean"] = float(bur.mean())
    out[f"{p}_iat_mean_min"] = float(iam.min()); out[f"{p}_iat_std_max"] = float(ias.max())
    for k, nm in enumerate(agg_named):
        out[f"{p}_prior_{nm}_mean_max"] = float(fm[:,k].max())
        out[f"{p}_prior_{nm}_max_max"]  = float(fx[:,k].max())
    return out

print("Computing 103 trajectory features (~30s)...")
t0 = time.time()
traj_rows = []
for tx_idx in range(N_tx):
    t_T = int(tx_time[tx_idx])
    in_w  = pick_top_wallets(incident_in.get(tx_idx,  []))
    out_w = pick_top_wallets(incident_out.get(tx_idx, []))
    in_summ  = [per_wallet_priors(w, t_T) for w in in_w]
    out_summ = [per_wallet_priors(w, t_T) for w in out_w]
    row = {}
    row.update(aggregate_side(in_summ,  "in",  t_T))
    row.update(aggregate_side(out_summ, "out", t_T))
    row["both_sides_have_illicit"] = float(
        row["in_n_wallets_with_illicit"]>0 and row["out_n_wallets_with_illicit"]>0)
    row["total_n_illicit_priors"]  = row["in_n_illicit_sum"]+row["out_n_illicit_sum"]
    row["total_n_wallets_with_illicit"] = row["in_n_wallets_with_illicit"]+row["out_n_wallets_with_illicit"]
    row["total_co_illicit"] = row["in_co_illicit_sum"]+row["out_co_illicit_sum"]
    row["min_recency_to_illicit"] = min(row["in_recency_to_illicit_min"], row["out_recency_to_illicit_min"])
    row["max_illicit_frac_either_side"] = max(row["in_illicit_frac_max"], row["out_illicit_frac_max"])
    row["max_decayed_illicit_either"]   = max(row["in_decayed_illicit_max"], row["out_decayed_illicit_max"])
    row["max_co_illicit_frac_either"]   = max(row["in_co_illicit_frac_max"], row["out_co_illicit_frac_max"])
    row["total_frac_first_appearance"] = (
        (row["in_frac_first_appearance"]*max(row["in_n_wallets"],1) +
         row["out_frac_first_appearance"]*max(row["out_n_wallets"],1)) /
        max(row["in_n_wallets"]+row["out_n_wallets"],1))
    T_btc = float(tx_X_full[tx_idx, total_btc_idx_full])
    max_p = max(row["in_prior_total_BTC_max_max"], row["out_prior_total_BTC_max_max"])
    mean_p = max(row["in_prior_total_BTC_mean_max"], row["out_prior_total_BTC_mean_max"])
    row["T_btc_vs_max_prior"]  = T_btc/max(max_p,1.0)
    row["T_btc_vs_mean_prior"] = T_btc/max(mean_p,1.0)
    traj_rows.append(row)
traj_df = pd.DataFrame(traj_rows)
traj_X  = traj_df.values.astype(np.float32)
F_TRAJ = traj_X.shape[1]
print(f"  done: traj_X={traj_X.shape} ({time.time()-t0:.1f}s)")

Computing 103 trajectory features (~30s)...


  done: traj_X=(203769, 103) (60.8s)


In [6]:
labelled   = (tx_label != -1)
train_mask = labelled & (tx_time <= TRAIN_END)
test_mask  = labelled & (tx_time >  TRAIN_END)
y_train = tx_label[train_mask]; y_test = tx_label[test_mask]
test_t_arr = tx_time[test_mask]
print(f"train: n={int(train_mask.sum()):,}  illicit_rate={y_train.mean():.4f}")
print(f"test:  n={int(test_mask.sum()):,}  illicit_rate={y_test.mean():.4f}")

X_intr_tr = tx_X_intrinsic[train_mask]; X_intr_te = tx_X_intrinsic[test_mask]
traj_tr   = traj_X[train_mask];          traj_te   = traj_X[test_mask]
A_tr      = block_A_X[train_mask];       A_te      = block_A_X[test_mask]
B_tr      = block_B_X[train_mask];       B_te      = block_B_X[test_mask]

ablations = {
    "A0 [108 intrinsic only]":               (X_intr_tr, X_intr_te),
    "A1 [108 + 103 traj]":                   (np.concatenate([X_intr_tr, traj_tr],axis=1),
                                              np.concatenate([X_intr_te, traj_te],axis=1)),
    f"A2 [108 + traj + Block A ({F_A})]":    (np.concatenate([X_intr_tr, traj_tr, A_tr],axis=1),
                                              np.concatenate([X_intr_te, traj_te, A_te],axis=1)),
    f"A3 [108 + traj + Block B ({F_B})]":    (np.concatenate([X_intr_tr, traj_tr, B_tr],axis=1),
                                              np.concatenate([X_intr_te, traj_te, B_te],axis=1)),
    f"A4 [108 + traj + A + B ({F_A+F_B})]":  (np.concatenate([X_intr_tr, traj_tr, A_tr, B_tr],axis=1),
                                              np.concatenate([X_intr_te, traj_te, A_te, B_te],axis=1)),
}

results = {}
preds   = {}
for name, (Xtr, Xte) in ablations.items():
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                 n_jobs=-1, random_state=RANDOM_SEED)
    rf.fit(Xtr, y_train)
    yp  = rf.predict(Xte)
    ypr = rf.predict_proba(Xte)[:, 1]
    f1   = f1_score(y_test, yp, pos_label=1, zero_division=0)
    auc  = roc_auc_score(y_test, ypr)
    prauc= average_precision_score(y_test, ypr)
    results[name] = (f1, auc, prauc, Xtr.shape[1])
    preds[name]   = yp
    print(f"  [{name}]  dim={Xtr.shape[1]}  F1={f1:.4f}  AUC={auc:.4f}  PR-AUC={prauc:.4f}  ({time.time()-t0:.1f}s)")
    last_rf = rf

train: n=29,894  illicit_rate=0.1158
test:  n=16,670  illicit_rate=0.0650


  [A0 [108 intrinsic only]]  dim=108  F1=0.8021  AUC=0.9026  PR-AUC=0.7855  (3.4s)


  [A1 [108 + 103 traj]]  dim=211  F1=0.8098  AUC=0.9196  PR-AUC=0.8029  (3.3s)


  [A2 [108 + traj + Block A (21)]]  dim=232  F1=0.8114  AUC=0.9217  PR-AUC=0.8029  (3.1s)


  [A3 [108 + traj + Block B (17)]]  dim=228  F1=0.8122  AUC=0.9160  PR-AUC=0.8027  (3.1s)


  [A4 [108 + traj + A + B (38)]]  dim=249  F1=0.8100  AUC=0.9141  PR-AUC=0.8012  (2.8s)


In [7]:
print("=" * 75)
print(f"{'Model':45s}  {'dim':>5s}  {'F1':>7s}  {'AUC':>7s}  {'PR-AUC':>7s}")
print("=" * 75)
for name, (f1, auc, prauc, dim) in results.items():
    print(f"  {name:43s}  {dim:>5d}  {f1:>7.4f}  {auc:>7.4f}  {prauc:>7.4f}")

f1_a1 = results["A1 [108 + 103 traj]"][0]
print()
for k in ["A2 [108 + traj + Block A (21)]",
          "A3 [108 + traj + Block B (17)]",
          "A4 [108 + traj + A + B (38)]"]:
    if k in results:
        d = results[k][0] - f1_a1
        print(f"  Δ F1 vs A1: {k}  →  {d:+.4f}")

# Per-timestep F1
print("\n" + "=" * 75)
print("Per-timestep test F1 (illicit class)")
print("=" * 75)
names = list(ablations.keys())
header = f"{'t':>3}  {'n':>5}  {'illicit':>7}" + "".join(f"  {n[:18]:>18s}" for n in names)
print(header)
for t in range(TRAIN_END+1, N_TIME_STEPS+1):
    m = (test_t_arr == t)
    if not m.any(): continue
    yt = y_test[m]
    if yt.sum() == 0:
        line = f"{t:>3}  {int(m.sum()):>5}  {int(yt.sum()):>7}" + "".join("  "+f"{'NaN':>18s}" for _ in names)
    else:
        cells = []
        for n in names:
            f1t = f1_score(yt, preds[n][m], pos_label=1, zero_division=0)
            cells.append(f"  {f1t:>18.4f}")
        line = f"{t:>3}  {int(m.sum()):>5}  {int(yt.sum()):>7}" + "".join(cells)
    print(line)

# Top-30 importance from A4
print("\n" + "=" * 75)
print("Top-30 features in A4 with source tags")
print("=" * 75)
all_names = list(feat_cols_intrinsic) + list(traj_df.columns) + BLOCK_A_FEATS + BLOCK_B_FEATS
imp = last_rf.feature_importances_
order = np.argsort(-imp)[:30]
n_a, n_b, n_traj = 0, 0, 0
F_off_traj = F_INTRINSIC + F_TRAJ
F_off_A    = F_off_traj + F_A
for rank, i in enumerate(order, 1):
    if i < F_INTRINSIC:
        tag = ""
    elif i < F_off_traj:
        tag = "  (TRAJ)"; n_traj += 1
    elif i < F_off_A:
        tag = "  (BLOCK A)"; n_a += 1
    else:
        tag = "  (BLOCK B)"; n_b += 1
    print(f"  {rank:2d}.  {imp[i]:.4f}  {all_names[i]}{tag}")
print(f"\n  in top-30: {n_traj} TRAJ  /  {n_a} BLOCK A  /  {n_b} BLOCK B")

Model                                            dim       F1      AUC   PR-AUC
  A0 [108 intrinsic only]                        108   0.8021   0.9026   0.7855
  A1 [108 + 103 traj]                            211   0.8098   0.9196   0.8029
  A2 [108 + traj + Block A (21)]                 232   0.8114   0.9217   0.8029
  A3 [108 + traj + Block B (17)]                 228   0.8122   0.9160   0.8027
  A4 [108 + traj + A + B (38)]                   249   0.8100   0.9141   0.8012

  Δ F1 vs A1: A2 [108 + traj + Block A (21)]  →  +0.0016
  Δ F1 vs A1: A3 [108 + traj + Block B (17)]  →  +0.0025
  Δ F1 vs A1: A4 [108 + traj + A + B (38)]  →  +0.0002

Per-timestep test F1 (illicit class)
  t      n  illicit  A0 [108 intrinsic   A1 [108 + 103 traj  A2 [108 + traj + B  A3 [108 + traj + B  A4 [108 + traj + A
 35   1341      182              0.9607              0.9636              0.9580              0.9609              0.9607
 36   1708       33              0.9143              0.9143             